# Time series Classification

In [ ]:
import sys
import os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
import pandas as pd
from src.utils.labels_binary_classification import generate_ischemic_labels
import numpy as np
from tslearn.shapelets import LearningShapelets
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from tslearn.shapelets import grabocka_params_to_shapelet_size_dict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier


import matplotlib.pyplot as plt

from tslearn.preprocessing import TimeSeriesScalerMinMax

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    RocCurveDisplay
)

import stumpy

PROFILE_PATH = "../../data/processed/time_series_features.pkl"
DIAG_PATH = "../../data/raw/heart_diagnoses_1.csv"
LABELS_OUT = "../../data/processed/subject_to_labels_ischemic.csv"

df = pd.read_pickle(PROFILE_PATH)
df["subject_id"] = df["subject_id"].astype(str)

print("Patient profile loaded:", df.shape)

labels_df = generate_ischemic_labels(
    diagnoses_path=DIAG_PATH,
    output_path=LABELS_OUT
)
labels_df["subject_id"] = labels_df["subject_id"].astype(str)  # Ensure matching types

print("Labels generated:")
print(labels_df["label_ischemic"].value_counts())

# Remove ICD-derived features to avoid data leakage
ICD_MACRO_COLS = [
    "ACS_acute_ischemic",
    "Chronic_ischemic",
    "Heart_failure",
    "Valvular_nonrheumatic",
    "Myocarditis_or_cardiomyopathy",
    "Pericardial_disease",
    "Endocarditis",
    "Cardiac_arrest",
    "Arrhythmias_or_conduction",
]

clf_df = df.drop(columns=[c for c in ICD_MACRO_COLS if c in df.columns])

# Merge labels with features
clf_df = clf_df.merge(labels_df[["subject_id", "label_ischemic"]], on="subject_id", how="inner")

print("Final classification dataset:", clf_df.shape)
print(f"Label distribution:\n{clf_df['label_ischemic'].value_counts()}")

# Convert 'gender' to boolean 
clf_df['gender'] = clf_df['gender'].map({'M': 1, 'F': 0})

clf_df

In [ ]:
# Evaluation: train vs test

def evaluate_binary(model, Xtr, ytr, Xte, yte):
    # predizioni (0/1)
    ytr_pred = model.predict(Xtr)
    yte_pred = model.predict(Xte)

    # probabilità (serve per ROC-AUC)
    ytr_proba = model.predict_proba(Xtr)[:, 1]
    yte_proba = model.predict_proba(Xte)[:, 1]

    results = {
        "train_accuracy": accuracy_score(ytr, ytr_pred),
        "test_accuracy": accuracy_score(yte, yte_pred),
        "train_balanced_accuracy": balanced_accuracy_score(ytr, ytr_pred),
        "test_balanced_accuracy": balanced_accuracy_score(yte, yte_pred),
        "train_precision": precision_score(ytr, ytr_pred),
        "test_precision": precision_score(yte, yte_pred),
        "train_recall": recall_score(ytr, ytr_pred),
        "test_recall": recall_score(yte, yte_pred),
        "train_f1": f1_score(ytr, ytr_pred),
        "test_f1": f1_score(yte, yte_pred),
        "train_roc_auc": roc_auc_score(ytr, ytr_proba),
        "test_roc_auc": roc_auc_score(yte, yte_proba),
    }
    return results, (ytr_pred, yte_pred, ytr_proba, yte_proba)


## Features classification

### Logistic Regression

In [ ]:


TARGET = "label_ischemic"
DROP_COLS_METADATA = ['subject_id', 'hadm_id', 'smoothed_II', 'long_title', 'icd_code', 'pNN50', 'SDNN']

# Fix: flatten the list so all columns are passed as a single list
X = clf_df.drop(columns=[TARGET] + DROP_COLS_METADATA)
y = clf_df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30, #divido il dataset in 70% training e 30% test
    stratify=y,
    random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Train label distribution:\n", y_train.value_counts(normalize=True))
print("Test  label distribution:\n", y_test.value_counts(normalize=True))

In [ ]:

lr_clf = Pipeline(steps=[
    ("model", LogisticRegression(
        penalty="l2",          # regolarizzazione L2: 2 regularization: penalizza coefficienti grandi, riduce l’overfitting, è stabile numericamente
        C=1.0,                 # forza regolarizzazione (più piccolo = più reg)
        solver="lbfgs",
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

lr_clf.fit(X_train, y_train)

In [ ]:
lr_results, (y_train_pred, y_test_pred, y_train_proba, y_test_proba) = evaluate_binary(
    lr_clf, X_train, y_train, X_test, y_test
)

pd.Series(lr_results).sort_index()

In [ ]:
coef = lr_clf.named_steps["model"].coef_[0]
coef_df = (
    pd.Series(coef, index=X.columns)
    .sort_values(key=np.abs, ascending=False)
)

coef_df.head(10)
plt.figure(figsize=(8, 5))
coef_df.head(10).plot(kind="barh")
plt.title("Top-10 Logistic Regression Coefficients")
plt.gca().invert_yaxis()
plt.show()



In [ ]:
from sklearn.metrics import precision_recall_curve

prec, rec, thr = precision_recall_curve(y_test, y_test_proba)

plt.plot(thr, prec[:-1], label="Precision")
plt.plot(thr, rec[:-1], label="Recall")
plt.xlabel("Decision threshold")
plt.legend()
plt.show()


In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores_lr = cross_val_score(
    lr_clf,
    X_train,
    y_train,
    cv=5,
    scoring="balanced_accuracy"
)

print("Logistic Regression - 5-fold CV balanced accuracy")
print(f"Mean = {cv_scores_lr.mean():.3f}")
print(f"Std  = {cv_scores_lr.std():.3f}")


In [ ]:
# Confusion matrix (test)
cm = confusion_matrix(y_test, y_test_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot()
plt.title("Logistic Regression - Confusion Matrix (test)")
plt.show()

print("\nClassification report (test):")
print(classification_report(y_test, y_test_pred, target_names=["Non-ischemic (0)", "Ischemic (1)"]))

# ROC curve (test)

RocCurveDisplay.from_predictions(y_test, y_test_proba)
plt.title("Logistic Regression - ROC curve (test)")
plt.show()


## Support Vector Machine

In [ ]:
from sklearn.svm import SVC

# Create the SVM classifier
svm_clf = SVC(
    kernel="rbf",          # Radial Basis Function kernel
    C=1.0,                 # Regularization parameter
    gamma="scale",        # Kernel coefficient
    probability=True,     # Enable probability estimates
    class_weight="balanced",
    random_state=42
)

svm_clf.fit(X_train, y_train)

svm_results, (y_train_pred_svm, y_test_pred_svm, y_train_proba_svm, y_test_proba_svm) = evaluate_binary(
    svm_clf, X_train, y_train, X_test, y_test
)

pd.Series(svm_results).sort_index()

In [ ]:
prec, rec, thr = precision_recall_curve(y_test, y_test_proba)

plt.plot(thr, prec[:-1], label="Precision")
plt.plot(thr, rec[:-1], label="Recall")
plt.xlabel("Decision threshold")
plt.legend()
plt.show()

In [ ]:
# Plot the confusion matrix for SVM (test)
cm_svm = confusion_matrix(y_test, y_test_pred_svm, labels=[0, 1])
disp_svm = ConfusionMatrixDisplay(confusion_matrix=cm_svm, display_labels=[0, 1])
disp_svm.plot()
plt.title("SVM - Confusion Matrix (test)")
plt.show()

# PLot the ROC curve for SVM (test)
RocCurveDisplay.from_predictions(y_test, y_test_proba_svm)
plt.title("SVM - ROC curve (test)")
plt.show()

In [ ]:
cv_scores_svm = cross_val_score(
    svm_clf,
    X_train,
    y_train,
    cv=5,
    scoring="balanced_accuracy"
)

print("Logistic Regression - 5-fold CV balanced accuracy")
print(f"Mean = {cv_scores_lr.mean():.3f}")
print(f"Std  = {cv_scores_lr.std():.3f}")

## Gradient boosting Classifier

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
gb_clf = GradientBoostingClassifier(
    n_estimators=300,       
    learning_rate=0.01,    
    max_depth=6,              
    random_state=42
)

gb_clf.fit(X_train, y_train)

gb_results, (gb_y_train_pred, gb_y_test_pred, gb_y_train_proba, gb_y_test_proba) = evaluate_binary(
    gb_clf, X_train, y_train, X_test, y_test
)

print(pd.Series(gb_results).sort_index())



In [ ]:
cv_scores_gb = cross_val_score(
    gb_clf, X_train, y_train,
    cv=5,
    scoring="balanced_accuracy"
)
print("GB - 5-fold CV balanced accuracy")
print(f"Mean = {cv_scores_gb.mean():.3f} | Std = {cv_scores_gb.std():.3f}")

In [ ]:
# Confusion matrix (test)
cm_rf = confusion_matrix(y_test, gb_y_test_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=[0, 1])
disp.plot()
plt.title("GradientBoostingClassifier - Confusion Matrix (test)")
plt.show()

# ROC curve (test)
RocCurveDisplay.from_predictions(y_test, gb_y_test_proba)
plt.title("GradientBoostingClassifier - ROC curve (test)")
plt.show()

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(
    n_estimators=30, #un numero moderato di alberi è sufficiente per ridurre la varianza rispetto a un singolo decision tree.
    criterion="gini", 
    max_features=3, #Questo introduce randomizzazione tra gli alberi, riducendo la correlazione e migliorando la generalizzazione
    max_depth=4, #Limita la profondità degli alberi per prevenire overfitting,
    min_samples_split=2, # Valore standard che consente agli alberi di crescere senza eccessive restrizioni.
    min_samples_leaf=8, # Impedisce foglie troppo piccole → predizioni più stabili e generalizzabili.
    bootstrap=True,  
    class_weight="balanced",
    random_state=42
)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    "n_estimators": [30, 50, 100],
    "max_depth": [2, 4, 6, 8, None],
    "max_features": [2, 3, "sqrt"],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(5, 20),
    "bootstrap": [True],
    "criterion": ["gini", "entropy"],
    "class_weight": ["balanced", None]
}

In [ ]:
rf_base = RandomForestClassifier(random_state=42)

rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist,
    n_iter=30,
    scoring="balanced_accuracy",
    cv=5,
    n_jobs=-1,
    random_state=42
)

rf_search.fit(X_train, y_train)

print("Best parameters found:")
print(rf_search.best_params_)

In [ ]:
rf_best = rf_search.best_estimator_

rf_best.fit(X_train, y_train)

rf_best_results, (rfb_y_train_pred, rfb_y_test_pred,
                  rfb_y_train_proba, rfb_y_test_proba) = evaluate_binary(
    rf_best, X_train, y_train, X_test, y_test
)

pd.Series(rf_best_results).sort_index()


In [ ]:
# comparison with baseline RF and RF with tuned hyperparameters (RandomizedSearchCV)
rf_baseline = rf_clf  # il modello con parametri fissi
rf_baseline.fit(X_train, y_train)

rf_base_results, _ = evaluate_binary(
    rf_baseline, X_train, y_train, X_test, y_test
)

comparison_rf = pd.DataFrame({
    "Baseline RF": rf_base_results,
    "Tuned RF": rf_best_results
})

comparison_rf.sort_index()


In [ ]:
# Feature Importance for Random Forest
feat_importances = pd.Series(rf_best.feature_importances_, index=X.columns)
feat_importances.nlargest(10).plot(kind='barh', title='Feature Importance - Random Forest')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Confusion matrix (test)
cm_rf = confusion_matrix(y_test, rfb_y_test_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=[0, 1])
disp.plot()
plt.title("Random Forest - Confusion Matrix (test)")
plt.show()

print("\nClassification report (test):")
print(classification_report(
    y_test, rfb_y_test_pred,
    target_names=["Non-ischemic (0)", "Ischemic (1)"]
))

# ROC curve (test)
RocCurveDisplay.from_predictions(y_test, rfb_y_test_proba)
plt.title("Random Forest - ROC curve (test)")
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(7, 6))

RocCurveDisplay.from_predictions(
    y_test, y_test_proba,
    name="Logistic Regression",
    ax=ax
)

RocCurveDisplay.from_predictions(
    y_test, gb_y_test_proba,
    name="Gradient Boosting",
    ax=ax
)

RocCurveDisplay.from_predictions(
    y_test, rfb_y_test_proba,
    name="Random Forest",
    ax=ax
)

ax.plot([0, 1], [0, 1], "k--", label="Random guess")
ax.set_title("ROC Curves Comparison (Test set)")
ax.legend()
plt.show()


## Time series motifs

In [ ]:
from scipy.stats import skew
# Get the profile matrices of each patient
fs = 500
profile_matrices = []
patient_beats = {}
patient_anomalies = {}
features_list = []
# Use STUMPY to compute the matrix profile
m = int(0.8 * fs) 

for subject_id, data in df.groupby('subject_id'):
    signal = np.array(data['smoothed_II'].iloc[0])

    if len(signal) < m:
        print(f"Signal for subject {subject_id} is shorter than subsequence length m={m}. Skipping.")
        continue

    profile = stumpy.stump(signal, m)
    profile_matrices.append(profile)

    # Get the top motifs
    motif_distances, motif_indices = stumpy.motifs(signal, profile[:, 0], max_motifs=3, min_neighbors=3)
    patient_beats[subject_id] = motif_indices[0]

    # Anomaly detection using matrix profile (top 3 discords)
    discord_indices = np.argpartition(profile[:, 0], -3)[-3:]
    # Sort by anomaly score descending
    discord_indices = discord_indices[np.argsort(profile[discord_indices, 0])[::-1]]
    discord_scores = profile[discord_indices, 0]
    patient_anomalies[subject_id] = list(zip(discord_indices, discord_scores))

    # Features Extraction for classification
    P = profile[:, 0]
    I = profile[:, 1]

    P = np.asarray(P, dtype=np.float64)

    features = {
            'subject_id': subject_id,
            
            # --- Morphology Features (Profile P) ---
            'mp_min': np.min(P),       # Best Motif
            'mp_max': np.max(P),       # Worst Anomaly
            'mp_mean': np.mean(P),     # Average noise/irregularity
            'mp_std': np.std(P),       # Variance in morphology
            'mp_skew': skew(P),        # Is the profile lopsided?
            
            # --- Anomaly Count ---
            # Count how many segments are "very different" 
            'num_anomalies': np.sum(P > (np.mean(P) + 2*np.std(P))),
            
            # --- Rhythm Features (Indices I) ---
            # Calculate absolute distance to nearest neighbor
            # "How far do I have to look to find a match?"
            'avg_neighbor_dist': np.mean(np.abs(I - np.arange(len(I))))
        }
        
    features_list.append(features)




profile_matrices = np.array(profile_matrices)
profile_matrices.shape
    

In [ ]:
features_df = pd.DataFrame(features_list)
# Add the ischemic labels
features_df = features_df.merge(
    labels_df[["subject_id", "label_ischemic"]],
    on="subject_id",
    how="inner"
)

features_df

In [ ]:
# Visualize the top 3 anomalies for one specific patient
subject_to_plot = list(patient_beats.keys())[0] # Just pick the first patient
discords = patient_anomalies[subject_to_plot]  # List of (idx, score) tuples

# Retrieve that patient's signal again for plotting
signal = df[df['subject_id'] == subject_to_plot]['smoothed_II'].iloc[0]

plt.figure(figsize=(12, 5))
plt.plot(signal, color='lightgray', label="Full ECG")

# Highlight the top 3 anomalies in different colors
colors = ['red', 'orange', 'purple']
for i, (idx, score) in enumerate(discords):
    plt.plot(range(idx, idx+m), signal[idx : idx+m], color=colors[i % len(colors)], linewidth=2, label=f"Anomaly {i+1} (Score {score:.2f})")

plt.title(f"Subject {subject_to_plot}: Top 3 Anomalies")
plt.legend()
plt.show()

In [ ]:
indices = patient_beats[subject_to_plot]
plt.figure(figsize=(10, 5))
plt.title(f"Overlay of Dominant Motif for Subject {subject_to_plot}")

# Loop through every index found and plot that segment
for idx in indices:
    # Ensure we don't go out of bounds
    if idx + m < len(signal):
        plt.plot(signal[idx : idx+m], color='red', alpha=0.3) # alpha makes them transparent

plt.xlabel("Time (Samples)")
plt.ylabel("Amplitude")
plt.show()

In [ ]:
# plot the Matrix Profile directly under your ECG signal for the first patient
subject_id = df['subject_id'].iloc[0]
signal = np.array(df[df['subject_id'] == subject_id]['smoothed_II'].iloc[0])
profile = profile_matrices[0]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(signal, color='blue')
ax1.set_title(f'ECG Signal for Subject {subject_id}')
ax1.set_ylabel('Amplitude')
ax2.plot(profile[:, 0], color='orange')
ax2.set_title('Matrix Profile')
ax2.set_ylabel('Matrix Profile Value')
ax2.set_xlabel('Time')
plt.tight_layout()
plt.show()

In [ ]:
# Run a random forest classifier on the extracted features
TARGET = "label_ischemic"
X_feat = features_df.drop(columns=[TARGET, 'subject_id'])
y_feat = features_df[TARGET].astype(int)

# Split into train and test sets
Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    X_feat, y_feat,
    test_size=0.30,
    stratify=y_feat,
    random_state=42
)

# Normalize features
scaler_feat = StandardScaler()
Xf_train = scaler_feat.fit_transform(Xf_train)
Xf_test = scaler_feat.transform(Xf_test)

rf_clf = RandomForestClassifier(
    n_estimators=30, #un numero moderato di alberi è sufficiente per ridurre la varianza rispetto a un singolo decision tree.
    criterion="gini", 
    max_features=3, #Questo introduce randomizzazione tra gli alberi, riducendo la correlazione e migliorando la generalizzazione
    max_depth=4, #Limita la profondità degli alberi per prevenire overfitting,
    min_samples_split=2, # Valore standard che consente agli alberi di crescere senza eccessive restrizioni.
    min_samples_leaf=8, # Impedisce foglie troppo piccole → predizioni più stabili e generalizzabili.
    bootstrap=True,  
    class_weight="balanced",
    random_state=42
)

In [ ]:
from scipy.stats import randint
param_dist = {
    "n_estimators": [30, 50, 100],
    "max_depth": [2, 4, 6, 8, None],
    "max_features": [2, 3, "sqrt"],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(5, 20),
    "bootstrap": [True],
    "criterion": ["gini", "entropy"],
    "class_weight": ["balanced", None]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
rf_base = RandomForestClassifier(random_state=42)

rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist,
    n_iter=30,
    scoring="balanced_accuracy",
    cv=5,
    n_jobs=-1,
    random_state=42
)

rf_search.fit(Xf_train, yf_train)

print("Best parameters found:")
print(rf_search.best_params_)

In [ ]:
rf_best = rf_search.best_estimator_

rf_best.fit(Xf_train, yf_train)

rf_best_results, (rfb_y_train_pred, rfb_y_test_pred,
                  rfb_y_train_proba, rfb_y_test_proba) = evaluate_binary(
    rf_best, Xf_train, yf_train, Xf_test, yf_test
)

pd.Series(rf_best_results).sort_index()

In [ ]:
# comparison with baseline RF and RF with tuned hyperparameters (RandomizedSearchCV)
rf_baseline = rf_clf  # il modello con parametri fissi
rf_baseline.fit(Xf_train, yf_train)

rf_base_results, _ = evaluate_binary(
    rf_baseline, Xf_train, yf_train, Xf_test, yf_test
)

comparison_rf = pd.DataFrame({
    "Baseline RF": rf_base_results,
    "Tuned RF": rf_best_results
})

comparison_rf.sort_index()


In [ ]:
feat_importances = pd.Series(rf_best.feature_importances_, index=X_feat.columns)
feat_importances.nlargest(10).plot(kind='barh', title='Feature Importance - Random Forest')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Confusion matrix (test)
cm_rf = confusion_matrix(yf_test, rfb_y_test_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=[0, 1])
disp.plot()
plt.title("Random Forest - Confusion Matrix (test)")
plt.show()

print("\nClassification report (test):")
print(classification_report(
    yf_test, rfb_y_test_pred,
    target_names=["Non-ischemic (0)", "Ischemic (1)"]
))

# ROC curve (test)
RocCurveDisplay.from_predictions(yf_test, rfb_y_test_proba)
plt.title("Random Forest - ROC curve (test)")
plt.show()

## Time series shaplets Classification

In [ ]:
n_ts, ts_sz = clf_df.shape[0], clf_df['smoothed_II'].iloc[0].shape[0]
n_classes = clf_df['label_ischemic'].nunique()

In [ ]:
from sklearn.model_selection import train_test_split
from tslearn.preprocessing import TimeSeriesScalerMeanVariance

X = np.array(clf_df['smoothed_II'].tolist())
y = np.array(clf_df['label_ischemic'].tolist())


X = X.reshape(X.shape[0], X.shape[1], 1)


print(f"Original Shape: {X.shape}")
X = X[:, :1500, :]
print(f"New Shape (Time Sliced): {X.shape}") 

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Scale the data
scaler = TimeSeriesScalerMeanVariance()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Get the sampling rate from the data
sampling_rate_hz = 500 
shapelet_sizes = {
    20: 50,    # QRS Complex (40ms: sharp, distinctive)
    50: 30,    # P-Wave (100ms: atrial depolarization)
    100: 20,   # T-Wave (200ms: ventricular repolarization)
    200: 10,   # QT Interval (400ms: complete ventricular cycle)
}

print("Optimized ECG Sizes:", shapelet_sizes)

In [ ]:
shp_clf = LearningShapelets(n_shapelets_per_size=shapelet_sizes,
                        optimizer="adam",
                        weight_regularizer=.01,
                        max_iter=2000,
                        verbose=1)

In [ ]:
print(f"Min: {X_train.min()}, Max: {X_train.max()}")

In [ ]:
# Fit the shapelet model
shp_clf.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score
predicted_labels = shp_clf.predict(X_test)
print("Correct classification rate:", accuracy_score(y_test, predicted_labels))
predicted_locations = shp_clf.locate(X)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

ts_id = 0
n_shapelets = sum(shapelet_sizes.values())

# Set seaborn style
sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(14, 6), dpi=300)

# Plot the original time series
ax.plot(X[ts_id].ravel(), label='ECG Signal', color=sns.color_palette("deep")[0], alpha=0.7)

# Plot each shapelet at its matched location
colors = sns.color_palette("husl", n_colors=len(shp_clf.shapelets_))
for idx_shp, shp in enumerate(shp_clf.shapelets_):
    t0 = predicted_locations[ts_id, idx_shp]
    ax.plot(np.arange(t0, t0 + len(shp)), shp.ravel(), 
            linewidth=2.5, color=colors[idx_shp], 
            label=f'Shapelet {idx_shp + 1} (len={len(shp)})')

ax.set_title(f'Shapelet Matches on Time Series (ID={ts_id}) — {n_shapelets} shapelets extracted', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Time (samples)', fontsize=12)
ax.set_ylabel('Amplitude (scaled)', fontsize=12)


sns.despine()
plt.tight_layout()
plt.show()